# 04 — Simulate API v2 (this is the live-demo button)

The NHL EDGE-style API just shipped a new revision. Existing v1 keys are unchanged, but three new top-level keys now appear on every event:

- `expected_goals` — `double` — model output, only meaningful for shot events but always present
- `shot_quality_index` — `double`
- `puck_speed_mph` — `double`

Because bronze stores the payload as **VARIANT**, ingest **does not break** — the new keys just slip in. But:

1. The drift-detector view from notebook 03 will start returning rows
2. The DBSQL Alert watching that view will fire and email
3. The Lakehouse Monitor will see fresh rows in silver (existing columns) on its next refresh — useful for telling "how much v2 data has landed since the alert"

**Run this notebook live in front of the audience.** After it completes:

- Open the DBSQL Alert (the one created from `alerts/drift_alert.sql`) and click **Refresh** — it should flip to TRIGGERED
- Open Catalog Explorer → `silver.plays` → Quality tab to show the LHM dashboard

In [0]:
%pip install dotenv

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os, json, random, datetime as dt
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA     = os.getenv('UC_SCHEMA',  'hockey_schema_monitoring')
BRONZE_TABLE  = f'{UC_CATALOG}.{UC_SCHEMA}_bronze.plays_raw'
DRIFT_VIEW    = f'{UC_CATALOG}.{UC_SCHEMA}_silver.v_unknown_payload_keys'

N_EVENTS = int(os.getenv('EVENTS_PER_GAME', '400'))   # one extra game's worth of v2
SEED     = int(os.getenv('RANDOM_SEED', '42')) + 1
rng = random.Random(SEED)

print(f'Appending {N_EVENTS} v2 events to {BRONZE_TABLE}')

Appending 400 v2 events to alexander_booth.hockey_schema_monitoring_bronze.plays_raw


In [0]:
TEAMS  = ['TOR', 'BOS', 'NYR', 'EDM', 'COL', 'VGK', 'FLA', 'CAR', 'TBL', 'PIT']
EVENTS = ['shot-on-goal', 'goal', 'hit', 'faceoff', 'giveaway', 'takeaway',
          'blocked-shot', 'missed-shot', 'penalty']

def make_play_v2(n: int) -> dict:
    t_sec  = rng.randint(0, 20 * 60 - 1)
    period = rng.choice([1, 2, 3])
    mm, ss = divmod(20 * 60 - t_sec, 60)
    return {
        # v1 keys (unchanged)
        'game_id'     : f'202602{2000 + n // 200:04d}',
        'season'      : '20252026',
        'period'      : period,
        'period_time' : f'{mm:02d}:{ss:02d}',
        'event_type'  : rng.choice(EVENTS),
        'team_abbrev' : rng.choice(TEAMS),
        'player_id'   : 8470000 + rng.randint(0, 99999),
        'x_coord'     : rng.randint(-100, 100),
        'y_coord'     : rng.randint(-42, 42),
        'event_ts'    : (dt.datetime(2026, 5, 20, 20, 0)
                          + dt.timedelta(seconds=t_sec + (period - 1) * 20 * 60)
                       ).isoformat() + 'Z',
        # NEW v2 keys — these are what the drift detector will catch
        'expected_goals'      : round(rng.uniform(0.01, 0.45), 3),
        'shot_quality_index'  : round(rng.uniform(0.0, 1.0), 3),
        'puck_speed_mph'      : round(rng.uniform(40.0, 105.0), 1),
    }

rows = [{
    'source_version': 'v2',
    'payload_json'  : json.dumps(make_play_v2(n)),
} for n in range(N_EVENTS)]

print('Sample v2 payload (note three new keys):')
print(json.dumps(json.loads(rows[0]['payload_json']), indent=2))

Sample v2 payload (note three new keys):
{
  "game_id": "2026022000",
  "season": "20252026",
  "period": 2,
  "period_time": "18:42",
  "event_type": "hit",
  "team_abbrev": "CAR",
  "player_id": 8518480,
  "x_coord": 71,
  "y_coord": -30,
  "event_ts": "2026-05-20T20:21:18Z",
  "expected_goals": 0.209,
  "shot_quality_index": 0.498,
  "puck_speed_mph": 41.2
}


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField('source_version', StringType(), False),
    StructField('payload_json',   StringType(), False),
])
df = (spark.createDataFrame(rows, schema=schema)
           .withColumn('ingest_ts', F.current_timestamp())
           .withColumn('payload',   F.expr('parse_json(payload_json)'))
           .select('ingest_ts', 'source_version', 'payload'))

df.write.mode('append').saveAsTable(BRONZE_TABLE)
print(f'Appended {df.count():,} v2 rows. Bronze total:')
spark.sql(f'SELECT source_version, COUNT(*) AS n FROM {BRONZE_TABLE} GROUP BY source_version ORDER BY source_version').show()

Appended 400 v2 rows. Bronze total:
+--------------+----+
|source_version|   n|
+--------------+----+
|            v1|3200|
|            v2| 400|
+--------------+----+



In [0]:
print('Drift detector view, post-append:')
spark.sql(f'SELECT * FROM {DRIFT_VIEW}').show(truncate=False)
print('\nGo trigger the DBSQL Alert in the UI — it should fire on this result.')

Drift detector view, post-append:
+------------------+---------+--------------------------+--------------------------+------------------------+---------------+
|payload_key       |sightings|first_seen                |last_seen                 |source_versions_affected|source_versions|
+------------------+---------+--------------------------+--------------------------+------------------------+---------------+
|shot_quality_index|400      |2026-05-20 14:11:48.599922|2026-05-20 14:11:48.599922|1                       |[v2]           |
|puck_speed_mph    |400      |2026-05-20 14:11:48.599922|2026-05-20 14:11:48.599922|1                       |[v2]           |
|expected_goals    |400      |2026-05-20 14:11:48.599922|2026-05-20 14:11:48.599922|1                       |[v2]           |
+------------------+---------+--------------------------+--------------------------+------------------------+---------------+


Go trigger the DBSQL Alert in the UI — it should fire on this result.
